# 03 Fine-tuning LoRA


## 환경


In [ ]:
from pathlib import Path
import json
import os
import subprocess

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / ".env.example").exists():
  project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

raw_base_model_id = os.getenv("HF_BASE_MODEL")
if not raw_base_model_id:
  raise RuntimeError("HF_BASE_MODEL must be set in .env")
if raw_base_model_id.startswith("/absolute/path/"):
  raise RuntimeError("Replace HF_BASE_MODEL in .env with your local LM Studio model path")
base_model_path = Path(os.path.expandvars(os.path.expanduser(raw_base_model_id)))
if base_model_path.is_absolute() and not base_model_path.exists():
  raise RuntimeError(f"HF_BASE_MODEL path does not exist: {base_model_path}")
base_model_id = str(base_model_path)
data_dir = project_root / os.getenv("FINE_TUNE_DATA_DIR", "data/loyalty_qa_mlx")
adapter_dir = project_root / os.getenv("FINE_TUNE_ADAPTER_DIR", "outputs/qwen25-05b-myelite-adapter")
max_iters = int(os.getenv("FINE_TUNE_MAX_ITERS", "80"))
batch_size = int(os.getenv("FINE_TUNE_BATCH_SIZE", "1"))
num_layers = int(os.getenv("FINE_TUNE_NUM_LAYERS", "4"))
report_to = os.getenv("FINE_TUNE_REPORT_TO", "none")
wandb_project = os.getenv("WANDB_PROJECT", "chapter4-finetuning")
test_prompt = os.getenv(
  "FINE_TUNE_TEST_PROMPT",
  "[MyElite Loyalty Program FAQ]: What is the cost of the MyElite Loyalty Program?",
)

base_model_id, str(data_dir), str(adapter_dir), max_iters, report_to


## Dataset


In [ ]:
data_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "train.jsonl"
valid_path = data_dir / "valid.jsonl"
test_path = data_dir / "test.jsonl"

def to_chat_record(example):
  if "messages" in example:
    return example
  if {"prompt", "completion"}.issubset(example):
    return {
      "messages": [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]},
      ]
    }
  raise ValueError(f"Unsupported dataset row: {example}")

def write_jsonl(path, examples):
  with path.open("w", encoding="utf-8") as file:
    for example in examples:
      file.write(json.dumps(to_chat_record(example), ensure_ascii=False) + "\n")

def normalize_jsonl(path):
  examples = []
  with path.open("r", encoding="utf-8") as file:
    for line in file:
      if line.strip():
        examples.append(to_chat_record(json.loads(line)))
  write_jsonl(path, examples)
  return len(examples)

if train_path.exists() and valid_path.exists() and test_path.exists():
  print(f"use existing dataset: {data_dir}")
else:
  train_examples = [
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What is the cost of the MyElite Loyalty Program?",
      "completion": "The MyElite Loyalty Program costs 99 USD per year. This fee is non-refundable, even if you cancel the membership.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Can I cancel my MyElite Loyalty Program membership?",
      "completion": "Yes, you can cancel your membership at any time. You will continue to receive benefits until the end of your current subscription period.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Is the MyElite annual fee refundable?",
      "completion": "No. The 99 USD annual fee is non-refundable, including when the membership is canceled before the subscription period ends.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: How long do benefits continue after cancellation?",
      "completion": "Benefits continue until the end of the paid subscription period, even after cancellation.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What is the maximum cashback I can earn?",
      "completion": "MyElite members can earn up to 2% cashback on eligible purchases.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Does MyElite cashback apply to every purchase?",
      "completion": "No. Cashback applies only to eligible purchases under the MyElite Loyalty Program terms.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Can I transfer my MyElite membership to another person?",
      "completion": "No. MyElite membership is tied to the account holder and cannot be transferred to another person.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: When does a MyElite subscription renew?",
      "completion": "A MyElite subscription renews annually unless the member cancels before the renewal date.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What happens if I cancel before renewal?",
      "completion": "If you cancel before renewal, you will not be charged for the next year and your current benefits remain active until the subscription period ends.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Do I lose benefits immediately after cancellation?",
      "completion": "No. You keep MyElite benefits until the end of the subscription period that has already been paid for.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Is MyElite billed monthly or yearly?",
      "completion": "MyElite is billed yearly at 99 USD per year.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What should I say if the FAQ does not contain the answer?",
      "completion": "If the MyElite FAQ does not contain the answer, say that the information is not available in the FAQ instead of guessing.",
    },
  ]

  valid_examples = [
    {
      "prompt": "[MyElite Loyalty Program FAQ]: How much is MyElite each year?",
      "completion": "MyElite costs 99 USD per year.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: If I cancel today, do I get my money back?",
      "completion": "No. The annual fee is non-refundable, but benefits continue until the end of the subscription period.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What cashback rate can members receive?",
      "completion": "Members can earn up to 2% cashback on eligible purchases.",
    },
  ]

  test_examples = [
    {
      "prompt": "[MyElite Loyalty Program FAQ]: What is the cost of the MyElite Loyalty Program?",
      "completion": "The MyElite Loyalty Program costs 99 USD per year. The fee is non-refundable.",
    },
    {
      "prompt": "[MyElite Loyalty Program FAQ]: Can I cancel and still use benefits?",
      "completion": "Yes. After cancellation, benefits remain available until the current subscription period ends.",
    },
  ]

  write_jsonl(train_path, train_examples)
  write_jsonl(valid_path, valid_examples)
  write_jsonl(test_path, test_examples)

normalize_jsonl(train_path), normalize_jsonl(valid_path), normalize_jsonl(test_path)


## Command


In [3]:
def run_command(command):
  result = subprocess.run(
    command,
    cwd=project_root,
    text=True,
    capture_output=True,
  )
  if result.stdout:
    print(result.stdout)
  if result.returncode != 0:
    if result.stderr:
      print(result.stderr)
    raise RuntimeError(f"command failed: {' '.join(command)}")
  return result.stdout.strip()


## Fine-tuning 전


In [ ]:
before_output = run_command([
  "uv",
  "run",
  "--extra",
  "mlx",
  "mlx_lm.generate",
  "--model",
  base_model_id,
  "--prompt",
  test_prompt,
  "--max-tokens",
  "120",
])


## Train


In [ ]:
train_command = [
  "uv",
  "run",
  "--extra",
  "mlx",
  "mlx_lm.lora",
  "--model",
  base_model_id,
  "--train",
  "--data",
  str(data_dir),
  "--adapter-path",
  str(adapter_dir),
  "--iters",
  str(max_iters),
  "--batch-size",
  str(batch_size),
  "--num-layers",
  str(num_layers),
  "--mask-prompt",
]

if report_to != "none":
  print("wandb_project is enabled")
  train_command.extend(["--report-to", report_to, "--project-name", wandb_project])

print(" ".join(train_command))
train_output = run_command(train_command)


## Fine-tuning 후


In [6]:
after_output = run_command([
  "uv",
  "run",
  "--extra",
  "mlx",
  "mlx_lm.generate",
  "--model",
  base_model_id,
  "--adapter-path",
  str(adapter_dir),
  "--prompt",
  test_prompt,
  "--max-tokens",
  "120",
])


The MyElite Loyalty Program costs 99 USD per year.
Prompt: 49 tokens, 1457.619 tokens-per-sec
Generation: 15 tokens, 264.681 tokens-per-sec
Peak memory: 0.363 GB



## 비교


In [7]:
print("=== before fine-tuning ===")
print(before_output)
print("\n=== after fine-tuning ===")
print(after_output)


=== before fine-tuning ===
The cost of the MyElite Loyalty Program is determined by the level of loyalty and the number of points earned. The cost is based on the points earned and the number of points required to earn the next level of loyalty. The cost is not a one-time cost but a recurring cost that you will pay monthly or annually.

To get started, you can sign up for the MyElite Loyalty Program and then follow the onboarding process to earn points and start earning loyalty points. The cost of the MyElite Loyalty Program is not a one-time cost but a recurring cost that you will pay monthly
Prompt: 49 tokens, 1177.030 tokens-per-sec
Generation: 120 tokens, 275.607 tokens-per-sec
Peak memory: 0.357 GB

=== after fine-tuning ===
The MyElite Loyalty Program costs 99 USD per year.
Prompt: 49 tokens, 1457.619 tokens-per-sec
Generation: 15 tokens, 264.681 tokens-per-sec
Peak memory: 0.363 GB
